In [9]:
import numpy as np
import pandas as pd
from itertools import combinations
import torch

import sys
sys.path.append('../')
from utilities import binning_equal_lambdas, binning_equal_q

Z = 660.4581
AA = 'ARNDQEGHILKMPSTV'
AA7 = 'ARQEGLKMPTV'

In [2]:
def extract_fields_and_J_matrix(params):
    
    fields = params[:90]
    fields7 = params[90:100]
    fields = np.insert(fields,15*np.arange(6)+10, 0.0).reshape(6,16)
    fields7 = np.insert(fields7,6,0.0)
    params = params[100:]
    
    J_matrix = []
    J_matrix7 = []
    
    j = 0
    for i in combinations(range(7),2):
        if i[1] != 6:
            n = 15
            m = 15
            n_insert = 10
            m_insert = 10
            to_append = params[j:j+n*m].reshape(n,m)
            to_append = np.insert(to_append,n_insert, np.zeros(m), axis=0)
            to_append = np.insert(to_append,m_insert, np.zeros(n+1), axis=1)
            J_matrix.append(to_append)
            j = j + n*m
        else:
            n = 15
            m = 10
            n_insert = 10
            m_insert = 6
            to_append = params[j:j+n*m].reshape(n,m)
            to_append = np.insert(to_append,n_insert, np.zeros(m), axis=0)
            to_append = np.insert(to_append,m_insert, np.zeros(n+1), axis=1)
            J_matrix7.append(to_append)
            j = j + n*m
        print(i,j)
        
    J_matrix = np.array(J_matrix)
    J_matrix7 = np.array(J_matrix7)
    
    return fields, fields7, J_matrix, J_matrix7

def sorted_log10p_vector_from_fields_and_J_matrix(fields, fields7, J_matrix, J_matrix7):
    
    lengths = [torch.arange(16, dtype=torch.int8) for i in range(6)]
    lengths.append(torch.arange(11, dtype=torch.int8))
    all_seq = torch.cartesian_prod(*lengths)
    
    all_seq = all_seq.long()
    
    # generate p_vector
    p_vector = torch.zeros(11*16**6,dtype=torch.float32)

    for i in range(6):
        p_vector += fields[i,all_seq[:,i]]
        print(i)
    p_vector += fields7[all_seq[:,6]]
    
    j=0
    for i in combinations(range(6),2):
        p_vector += J_matrix[j,all_seq[:,i[0]],all_seq[:,i[1]]]
        print(i)
        j+=1
        
    for i in range(6):
        p_vector += J_matrix7[i,all_seq[:,i], all_seq[:,6]]
        print(i)
    
    p_vector = torch.exp(p_vector)
    
    p_vector /= Z
    print(p_vector.sum())
    
    return torch.log(p_vector).sort()[0] / np.log(10)

def log10q_vector_func(data, fields, fields7, J_matrix, J_matrix7):
    
    # generate p_vector
    q_vector = torch.zeros(len(data),dtype=torch.float32)

    for i in range(6):
        q_vector += fields[i,data[:,i]]
        print(i)
    q_vector += fields7[data[:,6]]
    
    j=0
    for i in combinations(range(6),2):
        q_vector += J_matrix[j,data[:,i[0]],data[:,i[1]]]
        print(i)
        j+=1
        
    for i in range(6):
        q_vector += J_matrix7[i,data[:,i], data[:,6]]
        print(i)
    
    q_vector = torch.exp(q_vector)
    
    q_vector /= Z
    print(q_vector.sum())
    
    return np.log10(q_vector.numpy())

In [3]:
#np.loadtxt('Dalkara_Potts_params.csv',delimiter=',')[101],np.loadtxt('Dalkara_Potts_params.csv',delimiter=',')[119], np.loadtxt('Dalkara_Potts_params.csv',delimiter=',')[1227]

In [4]:
#J_matrix[0,0,1], J_matrix[0,1,4], J_matrix7[0,0,2]

In [5]:
data = pd.read_csv('../data/Dalkara_letters.csv',index_col=0).query('T0 > 0')

transform_dict = {AA[i] : i for i in range(16)}
data.iloc[:,:6] = data.iloc[:,:6].replace(transform_dict)
transform_dict = {AA7[i] : i for i in range(11)}
data.iloc[:,6] = data.iloc[:,6].replace(transform_dict)

data = data.sort_values('T0', ascending=False)

counts = data['T0'].to_numpy()
data = data.iloc[:,:7].to_numpy()

/tmp/ipykernel_1640076/1602654513.py:4: FutureWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  data.iloc[:,:6] = data.iloc[:,:6].replace(transform_dict)
/tmp/ipykernel_1640076/1602654513.py:6: FutureWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  data.iloc[:,6] = data.iloc[:,6].replace(transform_dict)


In [6]:
fields, fields7, J_matrix, J_matrix7 = extract_fields_and_J_matrix(np.loadtxt('parameters_Potts/Dalkara_Potts_params.csv',delimiter=','))

(0, 1) 225
(0, 2) 450
(0, 3) 675
(0, 4) 900
(0, 5) 1125
(0, 6) 1275
(1, 2) 1500
(1, 3) 1725
(1, 4) 1950
(1, 5) 2175
(1, 6) 2325
(2, 3) 2550
(2, 4) 2775
(2, 5) 3000
(2, 6) 3150
(3, 4) 3375
(3, 5) 3600
(3, 6) 3750
(4, 5) 3975
(4, 6) 4125
(5, 6) 4275


In [7]:
sorted_log10p_vector = sorted_log10p_vector_from_fields_and_J_matrix(fields, fields7,J_matrix, J_matrix7)

0
1
2
3
4
5
(0, 1)
(0, 2)
(0, 3)
(0, 4)
(0, 5)
(1, 2)
(1, 3)
(1, 4)
(1, 5)
(2, 3)
(2, 4)
(2, 5)
(3, 4)
(3, 5)
(4, 5)
0
1
2
3
4
5
tensor(1.0000, dtype=torch.float64)


In [8]:
log10q_vector = log10q_vector_func(data, fields, fields7, J_matrix, J_matrix7)
argsort = np.argsort(log10q_vector)
sorted_log10q_vector = log10q_vector[argsort][::-1]
counts = counts[argsort][::-1]

0
1
2
3
4
5
(0, 1)
(0, 2)
(0, 3)
(0, 4)
(0, 5)
(1, 2)
(1, 3)
(1, 4)
(1, 5)
(2, 3)
(2, 4)
(2, 5)
(3, 4)
(3, 5)
(4, 5)
0
1
2
3
4
5
tensor(0.7877, dtype=torch.float64)


In [10]:
df_bins = binning_equal_q(sorted_log10p_vector, sorted_log10q_vector, counts, bins=200, writefolder=False)
df_bins.to_csv('df_bins_Dalkara_Potts.csv')

0
tensor(184549376) tensor(184543116)
elements in the bin: 6260
nonzeros: 6260
1
tensor(184543116) tensor(184536856)
elements in the bin: 6260
nonzeros: 6260
2
tensor(184536856) tensor(184530596)
elements in the bin: 6260
nonzeros: 6260
3
tensor(184530596) tensor(184524336)
elements in the bin: 6260
nonzeros: 6260
4
tensor(184524336) tensor(184518076)
elements in the bin: 6260
nonzeros: 6260
5
tensor(184518076) tensor(184511816)
elements in the bin: 6260
nonzeros: 6260
6
tensor(184511816) tensor(184505555)
elements in the bin: 6261
nonzeros: 6260
7
tensor(184505555) tensor(184499294)
elements in the bin: 6261
nonzeros: 6260
8
tensor(184499294) tensor(184493031)
elements in the bin: 6263
nonzeros: 6260
9
tensor(184493031) tensor(184486763)
elements in the bin: 6268
nonzeros: 6260
10
tensor(184486763) tensor(184480499)
elements in the bin: 6264
nonzeros: 6260
11
tensor(184480499) tensor(184474226)
elements in the bin: 6273
nonzeros: 6260
12
tensor(184474226) tensor(184467951)
elements in

elements in the bin: 42051
nonzeros: 6260
132
tensor(182647279) tensor(182604101)
elements in the bin: 43178
nonzeros: 6260
133
tensor(182604101) tensor(182560279)
elements in the bin: 43822
nonzeros: 6260
134
tensor(182560279) tensor(182515271)
elements in the bin: 45008
nonzeros: 6260
135
tensor(182515271) tensor(182467549)
elements in the bin: 47722
nonzeros: 6260
136
tensor(182467549) tensor(182418690)
elements in the bin: 48859
nonzeros: 6260
137
tensor(182418690) tensor(182369271)
elements in the bin: 49419
nonzeros: 6260
138
tensor(182369271) tensor(182317548)
elements in the bin: 51723
nonzeros: 6260
139
tensor(182317548) tensor(182262993)
elements in the bin: 54555
nonzeros: 6260
140
tensor(182262993) tensor(182207337)
elements in the bin: 55656
nonzeros: 6260
141
tensor(182207337) tensor(182150282)
elements in the bin: 57055
nonzeros: 6260
142
tensor(182150282) tensor(182090801)
elements in the bin: 59481
nonzeros: 6260
143
tensor(182090801) tensor(182029015)
elements in the 

In [11]:
df_bins

,edge_sx,edge_dx,n_lambdas,n_lambdas_inferred_aritm,n_lambdas_inferred_geo,n_nonzeros,mean_aritm_lambda,mean_geom_lambda,mean_lambda,var_uniform_lambda,var_lambda,mean_count,var_count
0,-2.819845,-4.829754,6260,3.363900e+02,1.717876e+03,6260,10437.411637,2043.827320,567.323655307895,3.492078e+07,685138.8074435949,560.869169,605972.309561
1,-4.829754,-5.077456,6260,5.949894e+03,6.193480e+03,6260,158.148209,151.928330,148.8305019225382,6.428794e+02,613.5240323210684,150.313898,1953.414407
2,-5.077456,-5.224660,6260,6.227292e+03,6.316936e+03,6260,97.812173,96.424128,95.90112584311409,8.986962e+01,88.16291775855244,97.301118,798.126740
3,-5.224660,-5.334960,6260,6.286652e+03,6.337409e+03,6260,72.264696,71.685920,71.62965919383566,2.777168e+01,27.75415548273847,72.572364,524.318565
4,-5.334960,-5.423796,6260,6.239297e+03,6.271958e+03,6260,57.297164,56.998785,56.90461982480375,1.136785e+01,11.341410032554455,57.107668,379.924829
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,-9.236722,-9.364379,3182926,3.002506e+06,3.034992e+06,6260,0.006908,0.006834,0.0068440880635222585,3.387304e-07,3.3772201763295897e-07,0.006517,0.034862
196,-9.364379,-9.522805,4366443,4.207501e+06,4.277682e+06,6260,0.004998,0.004916,0.004927466582714965,2.710285e-07,2.694483417913479e-07,0.004817,0.026036
197,-9.522805,-9.741857,6822817,6.442878e+06,6.648855e+06,6260,0.003285,0.003184,0.0031993048093536124,2.194614e-07,2.1713596875016336e-07,0.003102,0.016693
198,-9.741857,-10.101858,13149234,1.195330e+07,1.299476e+07,6260,0.001777,0.001635,0.0016611652254285068,1.619404e-07,1.5809983207664274e-07,0.001615,0.008791
